# TAGS.CSV 

Este notebook documenta el análisis exploratorio y limpieza de **tags.csv** del dataset de MovieLens.


**Entrada**: tags.csv \
**Objetivos**: lectura, validación, limpieza y transformación \
**Salida**: tags_clean.parquet


## Descripción del proceso

**Análisis y comprensión**

El dataset contiene cuatro columnas: `userId`, identificador del usuario; `movieId`, identificador entero de la película; `tag`, la etiqueta en formato texto; y `timestamp`, la fecha de la etiqueta expresada en segundos, con valores comprendidos entre "1996-03-26" y "2018-09-26".

**Validación**

Se comprueba que no existan valores nulos y que las fechas estén dentro del rango entre "1996-03-26" y "2018-09-26".

**Limpieza**

No se permiten valores nulos, por lo que el registro afectado se eliminaría. Además, se unifican todas las etiquetas a minúsculas; una normalización más avanzada de los tags (por ejemplo, para su uso en técnicas de PLN) se realizará en pasos futuros del proyecto.

**Transformación**

Se convierte `timestamp` a formato datetime y se guarda el resultado en el archivo tags.parquet.

## Análisis y comprensión del dataset

Se cargan los datos de `tags.csv` y se revisan los tipos de cada columna.

In [1]:
import pandas as pd
import numpy as np

In [2]:
#Lectura del archivo
tags = pd.read_csv("../data/01_raw/movieLens/tags.csv")
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [3]:
tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 151.7 KB


Los tipos de datos son correctos y no hay valores nulos en ninguna columna. `timestamp` se mantiene como entero por ahora; su conversión a fecha se realiza en la transformación.

In [4]:
tags.dtypes

userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object

## Validación

Se comprueba que no existan valores nulos y que las fechas estén dentro del rango esperado.

Se comprueba la ausencia de valores nulos con `isna()` e `isnull()`, dos métodos equivalentes en pandas.

In [5]:
# no se observan valores na
tags.isna().any()

userId       False
movieId      False
tag          False
timestamp    False
dtype: bool

In [6]:
# no se observan valores null
tags.isnull().any()

userId       False
movieId      False
tag          False
timestamp    False
dtype: bool

Ambas comprobaciones confirman que no hay valores nulos en el dataset.

Validación de fechas dentro del rango de estudio, entre "1996-03-26" y "2018-09-26".

In [7]:
fechaInicio = pd.to_datetime("1996-03-26").timestamp()
fechaFin = pd.to_datetime("2018-09-26").timestamp()
tags[~tags['timestamp'].between(fechaInicio, fechaFin, inclusive = "both")]

,userId,movieId,tag,timestamp


El resultado está vacío, por lo que todos los registros tienen fechas dentro del rango esperado.

## Limpieza

La validación anterior no ha encontrado valores nulos ni fechas fuera de rango. Aun así, se aplica la eliminación de nulos como medida de seguridad, y se normalizan las etiquetas a minúsculas.

Se eliminan los valores nulos o NA en caso de existir. Como ya se ha comprobado que no hay ninguno, esta operación no elimina ningún registro en este dataset.

In [8]:
tags = tags.dropna()


Se unifican todas las etiquetas a minúsculas para evitar que la misma etiqueta se trate como distinta por diferencias de mayúsculas y minúsculas (por ejemplo, "Funny" y "funny").

In [9]:
tags['tag'] = tags['tag'].str.lower()
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,boxing story,1445715207
4,2,89774,mma,1445715200


## Transformación

Se convierte `timestamp` de segundos Unix a formato datetime y se guarda el resultado en `tags_clean.parquet`.

In [12]:
tags['timestamp'] = pd.to_datetime(tags['timestamp'], unit="s")
tags.to_parquet('../data/02_processed/tags_clean.parquet', index=False)